<a href="https://colab.research.google.com/github/vanamnagasaiVarshini/thiranex-password/blob/main/Vulnerabillity%20Scanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import argparse
import socket
import json
import urllib.request
import urllib.error
from urllib.parse import urlparse


# ==============================================================================
# ETHICAL REMINDER:
# This tool is for educational purposes only.
# Only scan networks and systems for which you have explicit permission to test.
# Unauthorized scanning of third-party systems may violate laws and policies.
# ==============================================================================


# Common ports to scan
COMMON_PORTS = {
    21: "FTP",
    22: "SSH",
    23: "Telnet",
    25: "SMTP",
    53: "DNS",
    80: "HTTP",
    110: "POP3",
    443: "HTTPS",
    3306: "MySQL",
    3389: "RDP"
}


# A small mock list of "outdated" software for educational purposes
KNOWN_OUTDATED_VERSIONS = {
    "nginx/1.14.0": {"risk": "Medium", "remediation": "Upgrade to a newer stable nginx version."},
    "apache/2.2.15": {"risk": "High", "remediation": "Upgrade Apache to 2.4.x or later."},
    "openssh_4.3": {"risk": "High", "remediation": "Upgrade to a newer OpenSSH version."},
}


def parse_args():
    parser = argparse.ArgumentParser(description="Simple Vulnerability Scanner (Educational)")
    parser.add_argument("-t", "--target", required=True, help="Target IP address or URL (e.g., 127.0.0.1 or http://example.com)")
    parser.add_argument("-o", "--output", default="report.json", help="Output report file (default: report.json)")
    return parser.parse_args()


def check_open_ports(target_host, ports):
    open_ports = []
    print(f"[*] Scanning {target_host} for common open ports...")
    for port, service in ports.items():
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(1.0)
        try:
            result = sock.connect_ex((target_host, port))
            if result == 0:
                open_ports.append({"port": port, "service": service})
                print(f"    [+] Port {port} ({service}) is OPEN")
        except socket.error:
            pass
        finally:
            sock.close()
    return open_ports


def get_hostname_from_target(target):
    if "://" in target:
        parsed = urlparse(target)
        return parsed.hostname
    return target


def inspect_http_headers(target):
    findings = []
    if not target.startswith("http"):
        url = f"http://{target}"
    else:
        url = target

    print(f"[*] Inspecting HTTP headers for {url}...")
    try:
        req = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(req, timeout=3) as response:
            server_header = response.getheader("Server")
            if server_header:
                print(f"    [+] Server Header found: {server_header}")
                findings.append({"type": "Server Header", "value": server_header})

                server_lower = server_header.lower()
                for known_version, info in KNOWN_OUTDATED_VERSIONS.items():
                    if known_version in server_lower:
                        print(f"    [!] Outdated Software Detected: {server_header} (Matches {known_version})")
                        findings.append({
                            "type": "Vulnerability",
                            "description": f"Outdated software matched: {server_header}",
                            "risk": info["risk"],
                            "remediation": info["remediation"]
                        })
    except urllib.error.URLError as e:
        print(f"    [-] Could not connect to {url} for banner grabbing: {e}")
    except Exception as e:
        print(f"    [-] Error during HTTP inspection: {e}")

    return findings


def generate_report(target, open_ports, http_findings, output_file):
    report = {
        "target": target,
        "open_ports": open_ports,
        "findings": http_findings
    }
    with open(output_file, 'w') as f:
        json.dump(report, f, indent=4)
    print(f"\n[*] Scan complete. Report saved to {output_file}")


def main():
    args = parse_args()
    target = args.target
    host = get_hostname_from_target(target)

    try:
        resolved_ip = socket.gethostbyname(host)
        print(f"[*] Target {host} resolved to {resolved_ip}")
        target_ip = resolved_ip
    except socket.gaierror:
        print(f"[-] Could not resolve hostname {host}. Proceeding anyway...")
        target_ip = host

    open_ports = check_open_ports(target_ip, COMMON_PORTS)
    http_findings = inspect_http_headers(target)
    generate_report(target, open_ports, http_findings, args.output)

In [2]:
import sys

# Change this to your test target:
# - IP address: e.g., "scanme.nmap.org" (allowed for nmap demos)
# - Local: "127.0.0.1" if you are running a local server
your_target = "scanme.nmap.org"   # example

# Emulate:  python scanner.py -t your_target
sys.argv = ["scanner.py", "-t", your_target, "-o", "report.json"]

# Now call main(); output will be exactly like your terminal
main()

[*] Target scanme.nmap.org resolved to 45.33.32.156
[*] Scanning 45.33.32.156 for common open ports...
    [+] Port 22 (SSH) is OPEN
    [+] Port 80 (HTTP) is OPEN
[*] Inspecting HTTP headers for http://scanme.nmap.org...
    [+] Server Header found: Apache/2.4.7 (Ubuntu)

[*] Scan complete. Report saved to report.json


In [3]:
from google.colab import files
files.download("report.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>